In [1]:
import os
from pathlib import Path

import dotenv
import numpy as np
import pandas as pd

# Load environment variables
dotenv.load_dotenv()

# Set up output folder
OUTPUT_FOLDER = Path(os.environ["OUTPUT_FOLDER"])
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

In [2]:
OUTPUT_FOLDER

PosixPath('/Users/hoegner/GitHub/ar7_netzero_assessment/analysis/output_revision')

# 300: Prepare Scenarios for MAGICC Runs

This notebook prepares scenario batches for running climate projections with MAGICC.

## Overview

We restrict to a single universe: scenarios that achieve net-zero CO2 by 2090
(`achieve_netzero_co2` and `year_netzero_co2 <= 2090`). Every group below is built from
this same universe - there is no separate, independently-selected scenario set for the
Kyoto-GHG-extended variant (earlier versions of this notebook selected that group via
`achieve_netzero_ghg` alone, independent of the net-zero-CO2 universe; that's no longer
the case).

For each qualifying scenario we build up to three data variants ("groups"):

| Group | Description | Input File | Extra filter |
|-------|-------------|------------|--------------|
| Original | Harmonised, unextended emissions | Harmonised data | none |
| NZCO2 extended | Emissions flatlined after net-zero CO2 (stylised "NZCO2-hold" variant) | NZCO2 extended | none |
| NZGHG extended | Emissions flatlined after net-zero Kyoto GHG (stylised "NZGHG-hold" variant) | NZKyoto extended | scenario must *also* achieve net-zero Kyoto GHG (`achieve_netzero_ghg`) - not every net-zero-CO2 scenario reaches net-zero GHG too, so this group is a subset of the universe |

One row per (group, scenario) - **not** per (group, scenario, MAGICC flag) as in earlier
versions of this notebook. `301_run_magicc.ipynb` now always runs all three MAGICC
passes it needs (`ANTHROPOGENIC` harvest -> `CO2`-only QEXTRA -> Kyoto-basket QEXTRA) in
a single execution per row, since all three are needed for every scenario/variant
anyway, and the `ANTHROPOGENIC` harvest pass already produces the `ANTHROPOGENIC`
flag's own GSAT result as a side effect - no separate native `ANTHROPOGENIC`-only run is
needed any more. This cuts total MAGICC passes by ~40% versus the old
one-row-per-flag design (5 passes/scenario -> 3) and removes the flag axis from this
batch file entirely. See `301_run_magicc.ipynb`'s own documentation and
`notes/magicc-kyoto-flag-qextra-equivalence.md` / `notes/magicc-carbon-cycle-feedback-mechanics.md`
for the QEXTRA methodology itself.

Note: unifying the universe means one scenario that was previously included in the old
"Batch E" (achieved net-zero GHG in 2100, but net-zero CO2 only in 2091 - just missing
the <=2090 cutoff) is no longer included anywhere in this file.

## Inputs
- `101_netzero_timings.csv`: Net-zero timing metadata
- `100_harmonised_infilled_data.csv`: Harmonised and infilled emissions data (SCI v1.1, centrally harmonized/infilled - see `100_prepare_sci_data.ipynb`)
- `101_netzero_co2_extended.csv`: Emissions extended from net-zero CO2
- `101_netzero_kyoto_extended.csv`: Emissions extended from net-zero Kyoto GHG

## Output
- `300_all_batches_{VERSION}.csv`: Combined batch configuration file (columns: `MODEL`, `SCENARIO`, `ENSEMBLE_MEMBERS`, `FILE` - no `MAGICC_FLAG` column any more)

## Constants

In [3]:
ENSEMBLE_MEMBERS = 600
INPUT_FILE = OUTPUT_FOLDER / "101_netzero_timings.csv"
HARMONISED_DATA_FILE = OUTPUT_FOLDER / "100_harmonised_infilled_data.csv"
NETZERO_EXTENDED_FILE = OUTPUT_FOLDER / "101_netzero_co2_extended.csv"
NETZERO_EXTENDED_FILE_GHG = OUTPUT_FOLDER / "101_netzero_kyoto_extended.csv"
VERSION = "v0.6"

In [4]:
df = pd.read_csv(INPUT_FILE)

## Load and Filter Data

Check the total number of scenarios in the input data.

In [5]:
n_total = len(df)
print(f"There are {n_total} scenarios")

There are 1543 scenarios


In [6]:
df

,model,scenario,year_netzero_co2,year_netzero_kyoto_ar6gwp100,achieve_netzero_co2,achieve_netzero_ghg
0,AIM/CGE 2.0,SSP1-19,2056.0,2069.0,True,True
1,AIM/CGE 2.0,SSP1-26,NaN,NaN,False,False
2,AIM/CGE 2.0,SSP1-34,NaN,NaN,False,False
3,AIM/CGE 2.0,SSP1-45,NaN,NaN,False,False
4,AIM/CGE 2.0,SSP1-Baseline,NaN,NaN,False,False
...,...,...,...,...,...,...
1538,WITCH-GLOBIOM 4.4,CD-LINKS-NPi,NaN,NaN,False,False
1539,WITCH-GLOBIOM 4.4,CD-LINKS-NPi2020_1000,2076.0,2087.0,True,True
1540,WITCH-GLOBIOM 4.4,CD-LINKS-NPi2020_1600,2095.0,NaN,True,False
1541,WITCH-GLOBIOM 4.4,CD-LINKS-NPi2020_400,2055.0,2069.0,True,True


Filter for scenarios that achieve net-zero CO2 emissions and set the model-scenario index.

In [7]:
df_for_initial_batches = (
    df[df["achieve_netzero_co2"]]
    .set_index(["model", "scenario"])
)

In [8]:
orig_scens = len(df_for_initial_batches)

In [9]:
omit_due_to_late_nz = df_for_initial_batches["year_netzero_co2"][
    df_for_initial_batches["year_netzero_co2"]>2090
]
len(omit_due_to_late_nz)

80

In [10]:
df_for_initial_batches = df_for_initial_batches[
    df_for_initial_batches["year_netzero_co2"] <= 2090
]
print(
    f"Original: {orig_scens}, Omitted: {len(omit_due_to_late_nz)}, Now considered: {orig_scens - len(omit_due_to_late_nz)}"
)

Original: 864, Omitted: 80, Now considered: 784


## Create Batch Dataframes

### Group 1: Original (unextended) scenarios

In [11]:
df_batch_original = (
    df_for_initial_batches
    .copy()
    .assign(
        ENSEMBLE_MEMBERS=ENSEMBLE_MEMBERS,
        FILE=HARMONISED_DATA_FILE,
    )
    .reset_index()
    .rename(columns={"model": "MODEL", "scenario": "SCENARIO"})
    .drop(columns=[
        "year_netzero_co2",
        "year_netzero_kyoto_ar6gwp100",
        "achieve_netzero_co2",
        "achieve_netzero_ghg"
    ])
)
df_batch_original

,MODEL,SCENARIO,ENSEMBLE_MEMBERS,FILE
0,AIM/CGE 2.0,SSP1-19,600,/Users/hoegner/GitHub/ar7_netzero_assessment/a...
1,AIM/CGE 2.0,SSP2-19,600,/Users/hoegner/GitHub/ar7_netzero_assessment/a...
2,AIM/CGE 2.0,SSP2-26,600,/Users/hoegner/GitHub/ar7_netzero_assessment/a...
3,AIM/CGE 2.0,SSP3-34,600,/Users/hoegner/GitHub/ar7_netzero_assessment/a...
4,AIM/CGE 2.0,SSP4-26,600,/Users/hoegner/GitHub/ar7_netzero_assessment/a...
...,...,...,...,...
779,WITCH-GLOBIOM 3.1,SSP4-26,600,/Users/hoegner/GitHub/ar7_netzero_assessment/a...
780,WITCH-GLOBIOM 4.4,CD-LINKS-INDC2030i_1600,600,/Users/hoegner/GitHub/ar7_netzero_assessment/a...
781,WITCH-GLOBIOM 4.4,CD-LINKS-NDC2030i_1000,600,/Users/hoegner/GitHub/ar7_netzero_assessment/a...
782,WITCH-GLOBIOM 4.4,CD-LINKS-NPi2020_1000,600,/Users/hoegner/GitHub/ar7_netzero_assessment/a...


### Group 2: Net-zero CO2 extended scenarios (stylised "NZCO2-hold" variant)

In [12]:
df_batch_nzco2_extended = df_batch_original.assign(FILE=NETZERO_EXTENDED_FILE)
df_batch_nzco2_extended["SCENARIO"] = df_batch_nzco2_extended["SCENARIO"] + "_NZCO2"
df_batch_nzco2_extended

,MODEL,SCENARIO,ENSEMBLE_MEMBERS,FILE
0,AIM/CGE 2.0,SSP1-19_NZCO2,600,/Users/hoegner/GitHub/ar7_netzero_assessment/a...
1,AIM/CGE 2.0,SSP2-19_NZCO2,600,/Users/hoegner/GitHub/ar7_netzero_assessment/a...
2,AIM/CGE 2.0,SSP2-26_NZCO2,600,/Users/hoegner/GitHub/ar7_netzero_assessment/a...
3,AIM/CGE 2.0,SSP3-34_NZCO2,600,/Users/hoegner/GitHub/ar7_netzero_assessment/a...
4,AIM/CGE 2.0,SSP4-26_NZCO2,600,/Users/hoegner/GitHub/ar7_netzero_assessment/a...
...,...,...,...,...
779,WITCH-GLOBIOM 3.1,SSP4-26_NZCO2,600,/Users/hoegner/GitHub/ar7_netzero_assessment/a...
780,WITCH-GLOBIOM 4.4,CD-LINKS-INDC2030i_1600_NZCO2,600,/Users/hoegner/GitHub/ar7_netzero_assessment/a...
781,WITCH-GLOBIOM 4.4,CD-LINKS-NDC2030i_1000_NZCO2,600,/Users/hoegner/GitHub/ar7_netzero_assessment/a...
782,WITCH-GLOBIOM 4.4,CD-LINKS-NPi2020_1000_NZCO2,600,/Users/hoegner/GitHub/ar7_netzero_assessment/a...


### Group 3: Net-zero GHG extended scenarios (stylised "NZGHG-hold" variant)

Subset of the same net-zero-CO2 universe that also achieves net-zero Kyoto GHG - not
an independently-selected scenario set (see notebook intro).

In [13]:
df_batch_nzghg_extended = (
    df_for_initial_batches[df_for_initial_batches["achieve_netzero_ghg"]]
    .copy()
    .assign(
        ENSEMBLE_MEMBERS=ENSEMBLE_MEMBERS,
        FILE=NETZERO_EXTENDED_FILE_GHG,
    )
    .reset_index()
    .rename(columns={"model": "MODEL", "scenario": "SCENARIO"})
    .drop(columns=[
        "year_netzero_co2",
        "year_netzero_kyoto_ar6gwp100",
        "achieve_netzero_co2",
        "achieve_netzero_ghg"
    ])
)
df_batch_nzghg_extended["SCENARIO"] = df_batch_nzghg_extended["SCENARIO"] + "_NZKyoto"
df_batch_nzghg_extended

,MODEL,SCENARIO,ENSEMBLE_MEMBERS,FILE
0,AIM/CGE 2.0,SSP1-19_NZKyoto,600,/Users/hoegner/GitHub/ar7_netzero_assessment/a...
1,AIM/CGE 2.0,SSP2-19_NZKyoto,600,/Users/hoegner/GitHub/ar7_netzero_assessment/a...
2,AIM/CGE 2.1,CD-LINKS-NPi2020_400_NZKyoto,600,/Users/hoegner/GitHub/ar7_netzero_assessment/a...
3,AIM/CGE V2.2,ENGAGE-Feasibility-1000/Institutional_NZKyoto,600,/Users/hoegner/GitHub/ar7_netzero_assessment/a...
4,AIM/CGE V2.2,ENGAGE-Feasibility-Maximum-Effort/Institutiona...,600,/Users/hoegner/GitHub/ar7_netzero_assessment/a...
...,...,...,...,...
382,WITCH-GLOBIOM 3.1,SSP1-19_NZKyoto,600,/Users/hoegner/GitHub/ar7_netzero_assessment/a...
383,WITCH-GLOBIOM 3.1,SSP4-19_NZKyoto,600,/Users/hoegner/GitHub/ar7_netzero_assessment/a...
384,WITCH-GLOBIOM 4.4,CD-LINKS-NDC2030i_1000_NZKyoto,600,/Users/hoegner/GitHub/ar7_netzero_assessment/a...
385,WITCH-GLOBIOM 4.4,CD-LINKS-NPi2020_1000_NZKyoto,600,/Users/hoegner/GitHub/ar7_netzero_assessment/a...


## Concatenate All Batches

In [14]:
df_all_batches = pd.concat(
    [
        df_batch_original,
        df_batch_nzco2_extended,
        df_batch_nzghg_extended,
    ],
    ignore_index=True,
)

df_all_batches

,MODEL,SCENARIO,ENSEMBLE_MEMBERS,FILE
0,AIM/CGE 2.0,SSP1-19,600,/Users/hoegner/GitHub/ar7_netzero_assessment/a...
1,AIM/CGE 2.0,SSP2-19,600,/Users/hoegner/GitHub/ar7_netzero_assessment/a...
2,AIM/CGE 2.0,SSP2-26,600,/Users/hoegner/GitHub/ar7_netzero_assessment/a...
3,AIM/CGE 2.0,SSP3-34,600,/Users/hoegner/GitHub/ar7_netzero_assessment/a...
4,AIM/CGE 2.0,SSP4-26,600,/Users/hoegner/GitHub/ar7_netzero_assessment/a...
...,...,...,...,...
1950,WITCH-GLOBIOM 3.1,SSP1-19_NZKyoto,600,/Users/hoegner/GitHub/ar7_netzero_assessment/a...
1951,WITCH-GLOBIOM 3.1,SSP4-19_NZKyoto,600,/Users/hoegner/GitHub/ar7_netzero_assessment/a...
1952,WITCH-GLOBIOM 4.4,CD-LINKS-NDC2030i_1000_NZKyoto,600,/Users/hoegner/GitHub/ar7_netzero_assessment/a...
1953,WITCH-GLOBIOM 4.4,CD-LINKS-NPi2020_1000_NZKyoto,600,/Users/hoegner/GitHub/ar7_netzero_assessment/a...


In [15]:
df_all_batches.to_csv(OUTPUT_FOLDER / f"300_all_batches_{VERSION}.csv")